# **EDA previo a la limpieza**

El objetivo es tener un primer acercamiento a los datasets con los que se estará trabajando. Identificar variables o puntos que puedan ser problemáticos en el proceso de limpieza.

**Setup y carga del crudo**

In [17]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DIR_RAW = Path.cwd().parent / "data" / "raw"
if not DIR_RAW.exists():
    DIR_RAW = Path.cwd() / "data" / "raw"

archivos = sorted(DIR_RAW.glob("diversificado_*.csv"))
partes = []
for f in archivos:
    parte = pd.read_csv(f, dtype=str)
    parte.insert(0, "ARCHIVO", f.stem)  # solo en memoria, no se escribe a data/raw/
    partes.append(parte)
df = pd.concat(partes, ignore_index=True)
print(f"{len(archivos)} archivos cargados -> {df.shape[0]} filas, {df.shape[1]} columnas")

23 archivos cargados -> 11867 filas, 18 columnas


**a. Numero de registros y variables**

In [18]:
cols_negocio = [c for c in df.columns if c != "ARCHIVO"]
print(f"Registros (filas): {df.shape[0]}")
print(f"Variables (columnas): {len(cols_negocio)}")

por_departamento = (
    df.groupby("DEPARTAMENTO", sort=False)
    .size()
    .rename("registros")
    .reset_index()
    .sort_values("registros", ascending=False)
)
por_departamento

Registros (filas): 11867
Variables (columnas): 17


,DEPARTAMENTO,registros
0,CIUDAD CAPITAL,2161
1,GUATEMALA,1908
12,SAN MARCOS,724
5,ESCUINTLA,708
13,HUEHUETENANGO,591
9,QUETZALTENANGO,551
17,PETEN,516
16,ALTA VERAPAZ,475
10,SUCHITEPEQUEZ,437
4,CHIMALTENANGO,435


A partir del análisis de la data se identificaron **11,867** establecimientos educativos de nivel **DIVERSIFICADO** registrados en el MINEDUC, los cuales son las instituciones con las que se estará trabajando a lo largo de este proyecto. Asimismo, cabe mencionar que cada uno de los archivos CSV, uno por departamento, cuenta con **17 variables**, las cuales se estarán discutiendo más a fondo en el siguiente inciso.

Por otro lado, la distribución por departamento evidencia una fuerte concentración en **Ciudad Capital**, con **2,161** establecimientos, y **Guatemala**, con **1,908**, que el MINEDUC maneja por separado, de tal forma que juntos suman poco más de un tercio de todos los establecimientos del país, mientras que el resto se reparte entre los 21 departamentos restantes.

**b. Tipo de dato de cada variable**

Para clasificar los tipos de datos de cada una de las variables, se estuvo realizando a partir de la naturaleza estadística de cada variable, es decir, cuantitativas para los valores numéricos con los que tiene sentido hacer operaciones, las cuales podemos distribuir en discretas, que son datos que se pueden contar, y continuas, que son mediciones; contra cualitativas, que es un tipo de variable que utilizamos para categorías o etiquetas, las cuales pueden ser nominales si no tienen un orden intrínseco, u ordinales si hay un orden jerárquico natural.

Cabe mencionar que aunque algunas de las variables se ven numéricas, no todas tienen sentido operacional, ya que algunas funcionan como identificadores, tal como lo son las variables de **código**, **distrito** y **teléfono**.

In [19]:
# El "tipo" relevante es la naturaleza estadistica de cada variable, no el
# dtype de almacenamiento (todo se cargo como texto con dtype=str a proposito).
# La clasificacion es conocimiento de dominio, por eso se declara explicitamente.
clasificacion = [
    ("CODIGO", "Cualitativa", "Nominal (identificador)", "Codigo unico del establecimiento; llave del registro (##-##-####-##).", "02-01-0027-46"),
    ("DISTRITO", "Cualitativa", "Nominal (identificador)", "Codigo del distrito educativo al que pertenece el establecimiento.", "02-01-0175"),
    ("DEPARTAMENTO", "Cualitativa", "Nominal", "Departamento donde se ubica (Ciudad Capital va aparte de Guatemala).", "EL PROGRESO"),
    ("MUNICIPIO", "Cualitativa", "Nominal", "Municipio; en la capital son zonas (ZONA 1..25).", "GUASTATOYA"),
    ("ESTABLECIMIENTO", "Cualitativa", "Nominal (texto libre)", "Nombre propio del centro educativo.", "COLEGIO ..."),
    ("DIRECCION", "Cualitativa", "Nominal (texto libre)", "Direccion fisica del establecimiento.", "BARRIO EL CALVARIO"),
    ("TELEFONO", "Cualitativa", "Nominal (identificador)", "Telefono(s) de contacto; numero de etiqueta, no cantidad.", "79450881"),
    ("SUPERVISOR", "Cualitativa", "Nominal (texto libre)", "Nombre del supervisor educativo asignado.", "CARLOTA E. ALBUREZ"),
    ("DIRECTOR", "Cualitativa", "Nominal (texto libre)", "Nombre del director del establecimiento.", "JOSE A. LOPEZ"),
    ("NIVEL", "Cualitativa", "Ordinal (constante aqui)", "Nivel escolar; constante (DIVERSIFICADO) por el filtro de extraccion.", "DIVERSIFICADO"),
    ("SECTOR", "Cualitativa", "Nominal", "Quien administra/financia el establecimiento.", "OFICIAL / PRIVADO"),
    ("AREA", "Cualitativa", "Nominal", "Area geografica; incluye 'SIN ESPECIFICAR' (no-dato).", "URBANA / RURAL"),
    ("STATUS", "Cualitativa", "Nominal", "Estado operativo del establecimiento.", "ABIERTA / CERRADA ..."),
    ("MODALIDAD", "Cualitativa", "Nominal", "Modalidad linguistica de ensenanza.", "MONOLINGUE / BILINGUE"),
    ("JORNADA", "Cualitativa", "Nominal", "Jornada en que opera; 'SIN JORNADA' actua como no-dato.", "MATUTINA / DOBLE"),
    ("PLAN", "Cualitativa", "Nominal", "Plan del servicio educativo.", "DIARIO(REGULAR)"),
    ("DEPARTAMENTAL", "Cualitativa", "Nominal", "Direccion departamental del MINEDUC que lo gestiona.", "EL PROGRESO"),
]

tipos_variables = pd.DataFrame(
    clasificacion,
    columns=["variable", "naturaleza", "subtipo", "descripcion", "ejemplo"],
).set_index("variable")

with pd.option_context("display.max_colwidth", None):
    display(tipos_variables)

print("Resumen por naturaleza:", tipos_variables["naturaleza"].value_counts().to_dict())

,naturaleza,subtipo,descripcion,ejemplo
variable,,,,
CODIGO,Cualitativa,Nominal (identificador),Codigo unico del establecimiento; llave del registro (##-##-####-##).,02-01-0027-46
DISTRITO,Cualitativa,Nominal (identificador),Codigo del distrito educativo al que pertenece el establecimiento.,02-01-0175
DEPARTAMENTO,Cualitativa,Nominal,Departamento donde se ubica (Ciudad Capital va aparte de Guatemala).,EL PROGRESO
MUNICIPIO,Cualitativa,Nominal,Municipio; en la capital son zonas (ZONA 1..25).,GUASTATOYA
ESTABLECIMIENTO,Cualitativa,Nominal (texto libre),Nombre propio del centro educativo.,COLEGIO ...
DIRECCION,Cualitativa,Nominal (texto libre),Direccion fisica del establecimiento.,BARRIO EL CALVARIO
TELEFONO,Cualitativa,Nominal (identificador),"Telefono(s) de contacto; numero de etiqueta, no cantidad.",79450881
SUPERVISOR,Cualitativa,Nominal (texto libre),Nombre del supervisor educativo asignado.,CARLOTA E. ALBUREZ
DIRECTOR,Cualitativa,Nominal (texto libre),Nombre del director del establecimiento.,JOSE A. LOPEZ


Resumen por naturaleza: {'Cualitativa': 17}


A continuación se describe qué representa cada variable y qué valores puede tomar, agrupadas a partir de su rol.

**Identificadores (códigos)**

- **CODIGO**: código único de cada establecimiento, con formato ##-##-####-##, el cual funciona como la llave del registro y no se repite.
- **DISTRITO**: código del distrito educativo al que pertenece el establecimiento.

---

**Ubicación geográfica y administrativa**

- **DEPARTAMENTO**: departamento del país, donde el MINEDUC separa Ciudad Capital, que son las zonas de la capital, de Guatemala, que es el resto del departamento.
- **MUNICIPIO**: municipio, que en la capital toma la forma de zonas, de ZONA 1 a ZONA 25.
- **DIRECCION**: dirección física del establecimiento, en texto libre.
- **DEPARTAMENTAL**: dirección departamental, es decir, la unidad administrativa del MINEDUC que gestiona el establecimiento.

---

**Nombres y personas (texto libre)**

- **ESTABLECIMIENTO**: nombre propio del centro educativo.
- **SUPERVISOR**: nombre del supervisor educativo asignado.
- **DIRECTOR**: nombre del director del establecimiento.
- **TELEFONO**: teléfono o teléfonos de contacto, que aunque es numérico funciona como una etiqueta.

---

**Atributos categóricos del servicio educativo**

- **NIVEL**: nivel escolar. Cabe mencionar que de forma general es ordinal, ya que párvulos va antes que primaria, básico y diversificado, pero en este caso es constante en DIVERSIFICADO por el filtro con que se extrajeron los datos.
- **SECTOR**: quién administra o financia el establecimiento, ya sea OFICIAL, PRIVADO, MUNICIPAL o COOPERATIVA.
- **AREA**: si es URBANA o RURAL, donde también aparece SIN ESPECIFICAR, que en realidad es un no-dato.
- **STATUS**: estado operativo, como ABIERTA, CERRADA TEMPORALMENTE o CERRADA DEFINITIVAMENTE, entre otros.
- **MODALIDAD**: si es MONOLINGUE o BILINGUE.
- **JORNADA**: jornada en que opera, como MATUTINA, VESPERTINA, DOBLE, NOCTURNA o INTERMEDIA, donde SIN JORNADA vuelve a actuar como un no-dato.
- **PLAN**: plan del servicio, como DIARIO(REGULAR), FIN DE SEMANA y sus variantes semipresenciales.

---

**c. Cantidad y porcentaje de valores faltantes por variable**

In [20]:
faltantes = df[cols_negocio].isna().sum().rename("faltantes")
faltantes_pct = (faltantes / len(df) * 100).round(2).rename("faltantes_%")
resumen_faltantes = (
    pd.concat([faltantes, faltantes_pct], axis=1)
    .sort_values("faltantes", ascending=False)
)
resumen_faltantes

,faltantes,faltantes_%
DIRECTOR,1732,14.60
TELEFONO,946,7.97
SUPERVISOR,535,4.51
DISTRITO,532,4.48
DIRECCION,76,0.64
ESTABLECIMIENTO,5,0.04
AREA,0,0.00
PLAN,0,0.00
JORNADA,0,0.00
MODALIDAD,0,0.00


A partir de la tabla anterior se puede observar que la mayoría de las variables se encuentran completas en todas las observaciones, tal como la dirección **departamental**, el **departamento** donde se encuentra la institución, el **municipio**, el **nivel** educativo hasta el que operan, que en este caso es el mismo para todos como se mencionó previamente, el **sector**, el **código** que las identifica, el **estatus**, la **modalidad**, la **jornada**, el **plan** y el **área**.

Sin embargo, encontramos cinco observaciones a las que les falta el **establecimiento**, que es el nombre del centro educativo, lo cual resulta alarmante, ya que esta variable debería formar parte de las que están completas, dado que es el identificador con el que cualquier persona podría dar con la institución. Asimismo, en la **dirección** vemos que hay 76 registros faltantes, mientras que en el **distrito** hay 532 que no lo tienen, probablemente porque no se maneja un distrito en la ubicación de ese centro educativo. Por otro lado, en el **teléfono** faltan 946, y puede que se trate de instituciones que no cuentan con los recursos necesarios para manejar ese medio de contacto, de tal forma que solo manejen el contacto físico entre personas. Finalmente, el **supervisor** y el **director**, con 535 y 1732 faltantes respectivamente, son datos que probablemente hagan falta por una mala recolección.

**d. Cantidad de valores unicos**

In [21]:
unicos = df[cols_negocio].nunique().rename("valores_unicos").sort_values(ascending=False)
unicos_df = unicos.reset_index().rename(columns={"index": "variable"})
print(f"CODIGO es llave unica: {df['CODIGO'].nunique() == len(df)} "
      f"({df['CODIGO'].nunique()} unicos / {len(df)} filas)")
unicos_df

CODIGO es llave unica: True (11867 unicos / 11867 filas)


,variable,valores_unicos
0,CODIGO,11867
1,DIRECCION,7438
2,TELEFONO,6571
3,ESTABLECIMIENTO,6312
4,DIRECTOR,5519
5,DISTRITO,1681
6,SUPERVISOR,1280
7,MUNICIPIO,352
8,DEPARTAMENTAL,26
9,DEPARTAMENTO,23


En cuanto a la cantidad de valores únicos, se logra identificar que el **código** es el identificador único asignado a cada una de las instituciones, ya que cuenta con 11,867 valores distintos, es decir, uno por cada observación. No obstante, llama la atención que en la **dirección** y el **teléfono** haya varios registros que no son únicos, sobre todo en el caso del teléfono, pues uno esperaría que cada institución tenga un teléfono propio. En cuanto a la dirección, es probable que existan ciertas instituciones que se encuentran cerca una de la otra, de tal forma que terminen compartiendo esta propiedad.

**e. Cantidad de registros duplicados exactos**

In [22]:
dup_filas = df[cols_negocio].duplicated().sum()
dup_codigo = df["CODIGO"].duplicated().sum()
print(f"Duplicados exactos (fila completa): {dup_filas}")
print(f"Duplicados por CODIGO (llave de negocio): {dup_codigo}")

if dup_codigo > 0:
    df[df["CODIGO"].duplicated(keep=False)].sort_values("CODIGO")

Duplicados exactos (fila completa): 0
Duplicados por CODIGO (llave de negocio): 0


Como punto destacable, identificamos que no hay duplicados exactos, ni en la fila completa ni por el **código**, que es su identificador único. Esto es algo bueno, ya que indica que no se tienen este tipo de errores dentro del dataset.

**f. Variables con valores fuera de dominio o inconsistentes**

In [23]:
cols_categoricas = ["NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]

# Se consolidan los dominios en una sola tabla para verlos con ancho uniforme.
filas = []
for c in cols_categoricas:
    vc = df[c].value_counts(dropna=False)
    for categoria, conteo in vc.items():
        filas.append((c, categoria, int(conteo), round(conteo / len(df) * 100, 2)))

dominios = pd.DataFrame(filas, columns=["variable", "categoria", "conteo", "porcentaje"])
with pd.option_context("display.max_rows", None):
    display(dominios)

,variable,categoria,conteo,porcentaje
0,NIVEL,DIVERSIFICADO,11867,100.00
1,SECTOR,PRIVADO,9891,83.35
2,SECTOR,OFICIAL,1499,12.63
3,SECTOR,COOPERATIVA,298,2.51
4,SECTOR,MUNICIPAL,179,1.51
5,AREA,URBANA,9461,79.73
6,AREA,RURAL,2403,20.25
7,AREA,SIN ESPECIFICAR,3,0.03
8,STATUS,ABIERTA,6860,57.81
9,STATUS,CERRADA TEMPORALMENTE,3006,25.33


En cuanto a las variables categóricas, se puede observar lo siguiente. Para el caso de la variable **nivel**, vale la pena mencionar que no hay diferencia alguna, ya que dado el filtro que se aplicó para la recolección de los datasets desde el MINEDUC, todos los registros iban a contar con el mismo nivel, que es DIVERSIFICADO.

En cuanto al **sector**, no se observan datos que salgan de lo esperado, ya que estos varían entre PRIVADO, OFICIAL, COOPERATIVA y MUNICIPAL, teniendo en todas las categorías una cantidad razonable de conteos. Por otro lado, en el **área** solamente tenemos tres registros como SIN ESPECIFICAR, que no están ni en URBANA ni en RURAL, lo cual sí puede destacarse como algo peculiar.

En cuanto al **estatus**, de igual forma hay una cantidad razonable de conteos para cada uno de los valores que toma la variable, que son ABIERTA, CERRADA TEMPORALMENTE, CERRADA DEFINITIVAMENTE, TEMPORAL TITULOS y TEMPORAL NOMBRAMIENTO. Para la **modalidad** no se ve nada alarmante, ya que solamente tenemos las dos categorías esperadas, que son MONOLINGUE y BILINGUE. Asimismo, en la **jornada** todos los posibles valores que toma tienen una cantidad de conteos razonable, en donde estos pueden ser DOBLE, VESPERTINA, MATUTINA, SIN JORNADA, NOCTURNA o INTERMEDIA.

Finalmente, en cuanto al **plan** sí resulta ligeramente curiosa la forma en que están distribuidos sus valores, ya que aquí encontramos la primera inconsistencia notoria. Tenemos tres categorías que son semipresenciales, una de FIN DE SEMANA, otra que explícitamente dice UN DÍA A LA SEMANA y otra que solamente dice SEMIPRESENCIAL sin especificar si es fin de semana o un día a la semana, la cual supongo que es un mixto, y adicional hay una cuarta entrada que dice SEMIPRESENCIAL (DOS DÍAS A LA SEMANA). Asimismo, tenemos SABATINO y DOMINICAL, en donde usualmente algunos de los que van a ser tomados como fin de semana son los sabatinos o los dominicales, mientras que el caso de MIXTO tal vez sí tiene sentido que se pueda clasificar así. No obstante, después aparecen un IRREGULAR y un INTERCALADO, lo cual llama la atención, pues sí hay ciertas categorías que resultan inconsistentes o poco claras. Es por esto que vale la pena discutirlo, y cabe mencionar que más adelante, cuando ya se estén haciendo transformaciones sobre los datasets, valdría la pena juntar SABATINO y DOMINICAL dentro de lo que se cuenta como FIN DE SEMANA.

**g. Variables con formatos inconsistentes**

In [24]:
import re

# 1) Barrido sistematico de formato en TODAS las variables, para no dejar
#    ninguna fuera: espacios en los bordes, dobles espacios, minusculas y digitos.
cols = [c for c in df.columns if c != "ARCHIVO"]
resumen_formato = pd.DataFrame({
    "no_nulos":       [int(df[c].notna().sum()) for c in cols],
    "espacios_borde": [int((df[c].dropna() != df[c].dropna().str.strip()).sum()) for c in cols],
    "doble_espacio":  [int(df[c].dropna().str.contains("  ").sum()) for c in cols],
    "con_minuscula":  [int(df[c].dropna().str.contains(r"[a-z]").sum()) for c in cols],
    "con_digito":     [int(df[c].dropna().str.contains(r"\d").sum()) for c in cols],
}, index=cols)
display(resumen_formato)

# 2) Variables tipo codigo: conformidad de patron
print("CODIGO fuera de ##-##-####-##:", int((~df["CODIGO"].dropna().str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$")).sum()))

d = df["DISTRITO"].dropna()
formato_distrito = pd.Series(
    ["largo (##-##-####)" if re.match(r"^\d{2}-\d{2}-\d{4}$", v)
     else "corto (##-###)" if re.match(r"^\d{2}-\d{3}$", v)
     else "incompleto/otro" for v in d]
).value_counts()
print("\nFormatos de DISTRITO:")
display(formato_distrito)
print("ejemplos incompletos:", list(d[~d.str.match(r"^\d{2}-\d{2}-\d{4}$|^\d{2}-\d{3}$")].unique())[:5])

# 3) TELEFONO: un solo numero de 8 digitos vs otros formatos
t = df["TELEFONO"].dropna()
ocho = t.str.match(r"^\d{8}$")
print(f"\nTELEFONO con 8 digitos exactos: {int(ocho.sum())} de {len(t)}; no conforme: {int((~ocho).sum())}")
display(t[~ocho].value_counts().head(8))


# 4) Placeholders/typos numericos en campos de nombres
print("DIRECTOR con digitos:", list(df["DIRECTOR"].dropna()[df["DIRECTOR"].dropna().str.contains(r"\d")].unique()))
print("SUPERVISOR con digitos:", list(df["SUPERVISOR"].dropna()[df["SUPERVISOR"].dropna().str.contains(r"\d")].unique()))

,no_nulos,espacios_borde,doble_espacio,con_minuscula,con_digito
CODIGO,11867,0,0,0,11867
DISTRITO,11335,0,0,0,11335
DEPARTAMENTO,11867,0,0,0,0
MUNICIPIO,11867,0,0,0,2161
ESTABLECIMIENTO,11862,0,0,0,298
DIRECCION,11791,0,0,10,8340
TELEFONO,10921,0,0,1,10921
SUPERVISOR,11332,0,0,0,3
DIRECTOR,10135,0,0,0,3
NIVEL,11867,0,0,0,0


CODIGO fuera de ##-##-####-##: 0

Formatos de DISTRITO:


largo (##-##-####)    6226
corto (##-###)        5039
incompleto/otro         70
Name: count, dtype: int64

ejemplos incompletos: ['01-', '10-', '17-']

TELEFONO con 8 digitos exactos: 10670 de 10921; no conforme: 251


TELEFONO
24328801-24329098             4
79486433-79480166             4
2325732, 2320075, 2307014     3
4431414, 3109992              3
78739432-79649696             3
77648506-45419234-41177068    3
79540830-79540909             3
22202870-73                   2
Name: count, dtype: int64

DIRECTOR con digitos: ['000000', '0000000', '0']
SUPERVISOR con digitos: ['EDNA ODILIA ACEVED0']


Para esta parte se realizó un barrido sistemático sobre todas las variables, buscando garantizar que no se estuviera dejando ninguna fuera del análisis. A partir de este se puede observar que, en términos generales, el dataset está bastante limpio en cuanto a formato, ya que ninguna variable presenta espacios en los bordes ni dobles espacios internos, y las variables categóricas (nivel, sector, área, estatus, modalidad, jornada y plan) se mantienen uniformes en mayúsculas, de tal forma que las inconsistencias de formato se concentran en unas pocas variables puntuales.

Primero, el **código** es la variable mejor formada, ya que las 11,867 observaciones calzan con el patrón ##-##-####-##, es decir, no hay ningún caso fuera de formato, lo cual tiene sentido pues es el identificador único del establecimiento. No obstante, el **distrito** sí mezcla dos formatos, uno largo (##-##-####) con 6,226 registros y uno corto (##-###) con 5,039, y adicional aparecen 70 registros incompletos como 01-, 10- o 17-, que básicamente son códigos truncados que no aportan información.

Por otro lado, el **teléfono** es la variable con mayor inconsistencia de formato, ya que de los 10,921 valores no nulos, 10,670 son un número simple de 8 dígitos, pero 251 no lo son, en donde encontramos varios números juntos dentro de una misma celda separados por guiones, comas, espacios o incluso una Y, tal como 22202870-73 o 25763, 26725 Y 21568, además de algunos números más cortos de 8 dígitos.

Finalmente, aunque los campos de texto como la **dirección** son heterogéneos por naturaleza, cabe mencionar que en el **director** y el **supervisor** aparecen valores que no corresponden a un nombre, como 0, 000000 o el caso de ACEVED0, donde el último carácter es un cero en lugar de una O, los cuales son placeholders o errores de digitación que funcionan como un dato faltante disfrazado. Dicho esto, todas estas inconsistencias son puntos que se tendrán que estandarizar más adelante en la etapa de limpieza, sobre todo la del teléfono y la del distrito.

**h. Identificacion de problemas potenciales de calidad de datos**

In [25]:
# Auditoria complementaria para H: senales que no se ven directas en A-G.
cols = [c for c in df.columns if c != "ARCHIVO"]

# a) Nulos disfrazados de valor (texto que en realidad representa un faltante)
tokens = ["SIN DATO", "SIN INFORMACION", "SIN ESPECIFICAR", "SIN JORNADA",
          "N/A", "NA", "NINGUNO", "NO TIENE", "NULL", "-", ".", "0", "000000", "0000000"]
print("Nulos disfrazados por variable:")
for c in cols:
    v = df[c].dropna()
    hits = {t: int((v == t).sum()) for t in tokens if (v == t).any()}
    if hits:
        print(f"  {c}: {hits}")

# b) Granularidad: un establecimiento fisico puede tener varios CODIGO. Se agrupa por
#    nombre + direccion + departamento + municipio para no colapsar institutos con
#    nombre generico ubicados en municipios distintos.
g = (df.dropna(subset=["ESTABLECIMIENTO", "DIRECCION"])
       .groupby(["ESTABLECIMIENTO", "DIRECCION", "DEPARTAMENTO", "MUNICIPIO"])["CODIGO"].nunique())
print(f"\nGrupos (nombre+direccion+municipio) con mas de un CODIGO: {int((g > 1).sum())}; "
      f"maximo en uno solo: {int(g.max())}; grupos totales (aprox. instituciones): {len(g)}")

# c) DEPARTAMENTAL es una unidad administrativa mas fina que DEPARTAMENTO
print(f"\nDEPARTAMENTO unicos: {df['DEPARTAMENTO'].nunique()} | "
      f"DEPARTAMENTAL unicos: {df['DEPARTAMENTAL'].nunique()}")

Nulos disfrazados por variable:
  DIRECCION: {'.': 5}
  DIRECTOR: {'SIN DATO': 28, '-': 16, '.': 14, '0': 1, '000000': 1, '0000000': 1}
  AREA: {'SIN ESPECIFICAR': 3}
  JORNADA: {'SIN JORNADA': 1099}

Grupos (nombre+direccion+municipio) con mas de un CODIGO: 1545; maximo en uno solo: 10; grupos totales (aprox. instituciones): 9307

DEPARTAMENTO unicos: 23 | DEPARTAMENTAL unicos: 26


A partir de todo lo revisado en los incisos anteriores y de una auditoría adicional sobre los 23 archivos, variable por variable, se identifican los siguientes problemas potenciales de calidad de datos. Para cada uno se indica la variable involucrada, el problema y lo que se debería hacer para abordarlo.

- **DIRECTOR:** es la variable con más ausencias, con 1,732 nulos que representan un 14.6%, y adicional concentra faltantes disfrazados como SIN DATO, guiones, puntos y ceros (0, 000000, 0000000). El problema es que el vacío está representado de muchas formas distintas, de tal forma que no se puede medir el faltante real. Lo que se debería hacer es unificar todos esos marcadores a un único valor nulo en la limpieza, sin imputar.

- **TELEFONO:** presenta 946 nulos y 251 valores que no son un número simple de 8 dígitos, en donde se meten varios números en una misma celda separados por guiones, comas, espacios o incluso una Y, tal como 22202870-73 o 25763, 26725 Y 21568, además de algunos números más cortos. Lo que se debería hacer es normalizar a un formato único, ya sea separando los múltiples números o quedándose con el principal, y marcar como nulo lo que sea irrecuperable.

- **DISTRITO:** tiene 532 nulos, mezcla dos formatos, uno largo (##-##-####) y uno corto (##-###), y adicional trae 70 códigos truncados como 01-, 10- o 17-. Lo que se debería hacer es estandarizar a un solo formato de código de distrito y convertir los truncados a nulo, ya que son irreparables.

- **SUPERVISOR:** cuenta con 535 nulos y al menos un error de digitación donde un nombre termina en cero en lugar de O, como ACEVED0. Lo que se debería hacer es marcar los nulos y corregir los nombres cuando el error sea evidente.

- **ESTABLECIMIENTO:** hay 5 registros sin el nombre del centro educativo, siendo este el identificador legible con el que cualquier persona ubica la institución. Lo que se debería hacer es revisar esos 5 casos de forma puntual y, de no poder recuperar el nombre, dejarlos como nulo y evaluar si el registro sigue siendo utilizable. Por ejemplo, poner un nombre genérico con base en la ubicación.

- **DIRECCION:** tiene 76 nulos y 5 registros con un punto como dirección, que es un faltante disfrazado, además de la heterogeneidad propia de un texto libre con abreviaturas y mayúsculas o minúsculas mezcladas. Lo que se debería hacer es convertir esos puntos a nulo, mientras que la normalización fina del texto queda como algo opcional según el análisis.

- **AREA:** incluye la categoría SIN ESPECIFICAR en 3 registros, que es un no-dato disfrazado de categoría válida, ya que un establecimiento necesariamente es urbano o rural. Por ello, al hacer las transformaciones se dejará como NaN, y no al revés, es decir, no se estandariza todo a SIN ESPECIFICAR, de tal forma que ese vacío no se cuente como una categoría más ni se confunda con un valor real.

- **JORNADA:** aparece la categoría SIN JORNADA en 1,099 registros, pero a diferencia del caso anterior, al cruzarla con PLAN se observa que casi todos corresponden a planes semipresenciales o a distancia (semipresencial fin de semana, un día o dos días a la semana, y a distancia), en donde efectivamente no existe una jornada fija, de tal forma que no es un faltante disfrazado sino un valor legítimo. Lo que se debería hacer es conservarla como una categoría válida y no convertirla a nulo, revisando únicamente los pocos casos que no calzan con ese patrón, como los 2 que aparecen con plan de fin de semana.

- **PLAN:** tiene 13 categorías solapadas e inconsistentes que además mezclan dos ideas distintas, el horario semanal (entre semana o fin de semana) y la modalidad de estudio (presencial, semipresencial o a distancia). Lo que se debería hacer al transformar es consolidarlas bajo un criterio propio, y la recomendación es dejar cuatro categorías. La primera sería ENTRE SEMANA, que agrupa DIARIO(REGULAR), el semipresencial de un día y de dos días a la semana, y el semipresencial que no especifica día, ya que su jornada es de entre semana. La segunda sería FIN DE SEMANA, que agrupa FIN DE SEMANA, el semipresencial de fin de semana, SABATINO y DOMINICAL. La tercera sería A DISTANCIA, que agrupa A DISTANCIA y VIRTUAL A DISTANCIA, ya que son una modalidad y no un horario, de tal forma que no se pueden forzar a entre semana o fin de semana. Y la cuarta sería MIXTO, únicamente para MIXTO, IRREGULAR e INTERCALADO, que son poco claros. Cabe mencionar que mandar el semipresencial de un día a la semana a entre semana es un supuesto, ya que la categoría no especifica cuál día es.

- **NIVEL:** es constante en DIVERSIFICADO por el filtro con que se extrajeron los datos, y el sufijo -46 del código lo confirma, de tal forma que no aporta variabilidad alguna. Lo que se debería hacer es conservarla como una constante documentada o descartarla para el análisis (dado el flujo de extracción se sabe que todas las observaciones son de DIVERSIFICADO).

- **DEPARTAMENTO y DEPARTAMENTAL:** el DEPARTAMENTO viene sin tildes, como PETEN o QUICHE, mientras que la dirección DEPARTAMENTAL viene con tildes, como PETÉN o QUICHÉ, y adicional esta última subdivide Guatemala en norte, sur, oriente y occidente, así como Quiché, razón por la cual hay 26 direcciones departamentales para 23 departamentos. Esto no es un error, pero lo que se debería hacer es estandarizar la acentuación y tener claro que DEPARTAMENTAL es una unidad administrativa más fina que DEPARTAMENTO.

- **CODIGO y granularidad del registro:** un mismo establecimiento, identificado por su nombre, dirección y municipio, puede tener varios códigos, ya que se encontraron 1,545 casos con más de uno y hasta 10 códigos para un mismo centro, como el CENTRO EDUCATIVO MAYA en Los Amates, Izabal, en donde los códigos difieren entre sí por la jornada o el plan autorizado. El problema es que contar filas no equivale a contar instituciones, pues las 11,867 filas corresponden a servicios autorizados y no a centros físicos, que rondan los 9,307 según esta aproximación. Lo que se debería hacer es deduplicar por establecimiento cuando se quiera saber cuántas instituciones hay, y usar el código cuando se quiera saber cuántos servicios o autorizaciones existen, teniendo en cuenta que esta agrupación por nombre es aproximada, ya que hay nombres genéricos que se repiten entre municipios distintos. Cabe mencionar que en la limpieza estas observaciones se mantienen separadas y no se fusionan, dado que difieren en atributos como la jornada o el plan, de tal forma que se puedan hacer análisis posteriores basados en si el servicio es entre semana o fin de semana.
